In [0]:
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.client import MlflowClient
from xgboost import XGBClassifier
import pickle

mlflow.autolog(disable=True)

In [0]:
%pip install "numpy==1.21.6" "scikit-learn==1.0.2" "xgboost==1.6.2" "imbalanced-learn==0.10.1" "skops"
dbutils.library.restartPython()

In [0]:
%run ../../config/utils

In [0]:
import joblib, xgboost as xgb

pkl_path = f"/Volumes/{catalog_name}/pe/helpers/bbm_propensity_model/XGBoost_Model.pkl"
grid = joblib.load(pkl_path)  

In [0]:
import json, xgboost as xgb
import pandas as pd

# Assume your fitted object is named `grid`
best_pipe = grid.best_estimator_
xgb_clf   = best_pipe.named_steps["clf"]            # the XGBClassifier

# Save the model in a version-stable XGBoost format (JSON or UBJSON).
# JSON is human-readable; UBJSON ('ubj') is smaller. Either works across versions.
xgb_clf.save_model(f"/Volumes/{catalog_name}/pe/helpers/bbm_propensity_model/XGBoost_Model.json")      # or "xgb_model.ubj"

# --- capture metadata needed at inference ---
# 1) Feature order used during training (prefer the original training DataFrame cols)
try:
    # if you trained on a pandas DataFrame:
    feature_names = list(best_pipe[:-1].transformer_list_)  # only if you had transformers
except Exception:
    # If you trained directly with a DataFrame and no transformers:
    # Replace `training_df` with the actual DataFrame used to fit.
    # feature_names = list(training_df.drop(columns=[label_column]).columns)
    pass

# Fallback: if you trained on numpy arrays, ensure you manually define `feature_names`
# so you can reorder columns consistently at inference time.

meta = {
    "feature_names": "",
    "classes": [str(i) for i in list(getattr(xgb_clf, "classes_", None))],          # for classifiers
    "best_params": grid.best_params_,
    "scoring_refit": grid.refit,                             # 'AUC' per your printout
    "xgb_params": xgb_clf.get_xgb_params(),                 # what was used
}

with open(f"/Volumes/{catalog_name}/pe/helpers/bbm_propensity_model/XGBoost_Model.meta.json", "w") as f:
    json.dump(meta, f, indent=2)
